# Task 1.1 — Core Contribution / Architecture (8 marks)

## Step-by-Step Method Description

The paper proposes a **Copula Mixture Model (CM)** for dependency-seeking clustering on multi-view data. Below is the method described step-by-step from input to final output.

### Step 1: Multi-view Data Input
- **Description**: The method takes as input two co-occurring datasets (views): a $p$-dimensional random vector $X = (X_1, \ldots, X_p)$ and a $q$-dimensional random vector $Y = (Y_1, \ldots, Y_q)$, with $n$ paired observations $(x_i, y_i)$.
- **Reference**: Section 2, first paragraph; Section 4.1.
- **Purpose**: These two views represent different sources of information about the same objects. The goal is to discover clusters that explain the *dependencies between views*, not just within-view structure.

### Step 2: Marginal Distribution Specification
- **Description**: For each dimension $j$ of each view, the method specifies and fits a univariate marginal distribution $F_j$ with density $f_j$. These can be *any* continuous distribution (e.g., Gaussian for unbounded data, Beta for data on $[0,1]$, Exponential for positive data). The marginals are specified with prior distributions on their parameters — for example, normal and inverse-gamma priors for Gaussian margins, or gamma priors for Beta shape parameters.
- **Reference**: Section 4.1 (model specification); Section 5.2 ("CM uses Gaussian marginals in the first view and beta marginals in the second view"); Table 1 for simulation parameters.
- **Purpose**: By modelling each margin separately, the method decouples the shape of individual feature distributions from the joint dependence structure. This is the key innovation over standard Gaussian mixtures which assume all margins are Gaussian.

### Step 3: Copula Transformation to Normal Scores
- **Description**: Each observed value $x_j$ is transformed to a *normal score* via $\tilde{x}_j = \Phi^{-1}(F_j(x_j))$, where $F_j$ is the fitted marginal CDF and $\Phi^{-1}$ is the inverse standard normal CDF. This applies the *probability integral transform* followed by the inverse normal quantile function. After transformation, each dimension is marginally standard normal regardless of its original distribution.
- **Reference**: Section 3 (Equation 8); the density formula $f(x) = |P|^{-1/2} \exp\left(-\frac{1}{2} \tilde{x}^T (P^{-1} - I) \tilde{x}\right) \prod_j f_j(x_j)$.
- **Purpose**: After this transformation, the dependence structure between dimensions is fully characterised by a correlation matrix $P$ of the normal scores. This is the essence of the Gaussian copula — it separates dependence from marginals.

### Step 4: Block-Diagonal Correlation Matrix for Dependency-Seeking
- **Description**: The correlation matrix $P$ of the normal scores is constrained to have a **block-diagonal structure**: $P = \begin{pmatrix} P_x & 0 \\ 0 & P_y \end{pmatrix}$, where $P_x$ captures within-View-1 correlations and $P_y$ captures within-View-2 correlations. The off-diagonal blocks are zero, meaning that *given the cluster assignment*, the two views are conditionally independent.
- **Reference**: Section 2, Equation (4); Section 3, last paragraph ("if $P$ has a block diagonal structure... the conditional independence of $X|Z$ and $Y|Z$ will be preserved").
- **Purpose**: This block structure is the core of dependency-seeking clustering. It forces *all inter-view dependence* to be explained by the cluster assignments alone, ensuring the discovered clusters have a meaningful interpretation as groups sharing similar cross-view dependencies.

### Step 5: Dirichlet Process Mixture (DPM)
- **Description**: The number of clusters is not fixed a priori. Instead, the model uses a *Dirichlet Process prior* (Ferguson, 1973) with concentration parameter $\alpha$ and base distribution $G_0$. This allows the model to automatically determine the number of clusters from the data. Each cluster $c$ has its own parameters $(\theta_c, P_c)$, where $\theta_c$ holds all marginal parameters and $P_c$ is the cluster-specific block-diagonal correlation matrix.
- **Reference**: Section 4.1, Equation for $f_{(X,Y)}(x,y)$ as an integral over the Dirichlet process.
- **Purpose**: The Bayesian nonparametric prior avoids the need to select the number of clusters beforehand, which is particularly important because model mismatch in Gaussian mixtures can lead to *overestimation* of the number of clusters.

### Step 6: MCMC Inference (Algorithms 1 and 2)
- **Description**: Inference is performed via Markov Chain Monte Carlo. **Algorithm 1** (based on Neal's Algorithm 8) iterates over data points to update cluster assignments: for each point, it proposes either joining an existing cluster or creating a new one, accepting the proposal via a Metropolis-Hastings step using the meta-Gaussian likelihood. **Algorithm 2** updates the within-cluster parameters: (a) marginal parameters $\theta$ via Metropolis-Hastings, (b) the latent normal scores $\tilde{X}, \tilde{Y}$ via Metropolis-Hastings, and (c) the block-diagonal correlation matrices $\Sigma_x, \Sigma_y$ via Gibbs sampling from an inverse-Wishart posterior.
- **Reference**: Algorithm 1 (page 5); Algorithm 2 (page 5); Section 4.2 ("Posterior inference").
- **Purpose**: The MCMC scheme jointly samples cluster assignments, marginal parameters, latent normal scores, and correlation matrices, enabling fully Bayesian inference. The use of latent normal scores makes the Gaussian copula framework computationally tractable.

### Step 7: Output — Cluster Assignments
- **Description**: After sufficient MCMC iterations, the method outputs cluster assignments for each data point. Points in the same cluster share similar *inter-view dependency patterns*. The clustering can be summarised by the modal assignment or by analysing the posterior distribution of assignments across MCMC samples.
- **Reference**: Section 5 (experiments); Figures 3-4 (clustering results evaluated via Adjusted Rand Index).
- **Purpose**: The final clusters represent groups of observations that exhibit similar patterns of dependence between the two views, which is the primary goal of dependency-seeking clustering.

## Final Summary Sentence

This paper solves the problem of **dependency-seeking clustering on non-Gaussian multi-view data** by introducing a Gaussian copula-based mixture model that separates marginal distribution modelling from dependence structure estimation; the authors claim this is superior to existing Gaussian mixture approaches because it avoids the model mismatch that forces Gaussian mixtures to create spurious extra clusters when the data margins are non-Gaussian (e.g., beta or exponential), thereby preserving a meaningful, interpretable cluster structure.